In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")

    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
    driver.find_element(By.ID, "password").send_keys("12345678")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)

    # The only date/date-range filter in the current frontend is the Reports
    # period switch (7/30/90 days, verified in ReportsPage.jsx). No date inputs exist.
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'Reports')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//button[text()='30 Days']")))
    wait.until(EC.invisibility_of_element_located((By.XPATH, "//*[text()='Loading reports...']")))
    time.sleep(2)

    def range_text():
        els = driver.find_elements(By.XPATH, "//div[contains(text(), '\u2192')]")
        return els[0].text.strip() if els else ""

    before = range_text()
    assert before, "Date-range label not found on Reports page."
    print("Initial range:", before)

    # Apply a different valid period using the real buttons
    driver.find_element(By.XPATH, "//button[text()='7 Days']").click()
    wait.until(EC.invisibility_of_element_located((By.XPATH, "//*[text()='Loading reports...']")))
    time.sleep(2)
    after = range_text()
    print("Range after filter:", after)
    assert after and after != before, f"Filter did not change displayed data (before={before!r}, after={after!r})."

    print("PASS: Date filter applied and data changed")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("37_date_filter_FAIL.png")
finally:
    driver.quit()